# Missing Data Imputation

using MICE (Multiple Imputation by Chained Equations)


TODO:

You'll have to implement the following new classes:

CONFIRM that LKJCholesky samples are correlation matrixes not covariance ones 
    for each backend
    i.e. diagonal of (sample @ sample.T) should all be 1s
    tensorflow
        yes! confirmed
    jax
        yes confirmed, though it's not exaaaaaaactly 1... but darn close 
    pytorch
        yes confirmed
LKJCholesky distribution (for prior for the below param)
    which deals with CORRELATION matrixes (so diags should all be 1?)
LKJCholeskyParameter parameter
    correlation matrix parameter
    which deals with CORRELATION matrixes (so diags should all be 1?)
    or, maybe just do one which generates the whole Covariance matrix?
UPDATE MultivariateNormal distribution to take tril kwarg
    either tril or cov
    if cov, use the backend cov parameterization directly
        i.e. torch tod.multivariate_normal.MultivariateNormal
        jax tfp_jax.distributions.MultivariateNormal (if that exists?)
        tensorflow tfd.MultivariateNormal (if that exists?)
    if tril, use the backend tril parameterization
        i.e. torch tod.multivariate_normal.MultivariateNormalTriL (if that exists?)
        jax tfp_jax.distributions.MultivariateNormalTriL
        tensorflow tfd.MultivariateNormalTriL
    that way you can pass in the tril from LKJCholeskyParameter.sample() directly

In [ ]:
class MultipleImputationByChainedEquations(pf.Model):

    def __init__(self, d: int, n_missing: int):
        self.mu = pf.Parameter(shape=d)
        self.covariance_tril = pf.WishartParameter(d=d, output_tril=True)
        # TODO: still need to confirm if the above outputs covariance vs correlation matrixes
        self.z = pf.Parameter(shape=n_missing)  # latent parameters for missing data

    def __call__(self):
        return pf.MultivariateNormal(
            loc=self.mu(),
            tril=self.covariance_tril(),
        )

    def log_likelihood(self, missing_ix, y):
        """This method treats what is usually the x parameter as the missing data 
        index matrix, and y as the data matrix.  The missing data matrix is the same
        size as y, but when an element has a >=0 value, that value is the ID of the
        missing data element)

        E.g.:

        y = [
            [0, NaN, 2],
            [3, 4, NaN],
            [NaN, 7, NaN],
        ]

        missing_ix = [
            [-1, 0, -1],
            [-1, -1, 1],
            [2, -1, 3],
        ]
        """
        # For missing data, sample from imputed data latent posteriors
        missing_mask = missing_ix >= 0
        safe_unobs_ix = tf.where(missing_mask, missing_ix, 0)
        unobs_y = self.z[safe_unobs_ix]

        # Combine observed and imputed data
        combined_x = tf.where(missing_mask, unobs_y, y)

        # Calculate log likelihood of observed + imputed data
        # w.r.t. underlying multivariate normal model
        log_likelihoods = self().log_prob(combined_x)
        return O.sum(log_likelihoods, axis=None)


y = # some data matrix with NaNs where there are missing data

# Generate a missing data index matrix
missing_ix = np.full_like(y, -1)
missing_ix[np.isnan(y)] = np.arange(np.isnan(y).sum())

# Create and fit the MICE model
mice_model = MultipleImputationByChainedEquations(d=y.shape[1], n_missing=np.isnan(y).sum())
mice_model.fit(missing_ix, y)